# Plant Disease Detection & Diagnosis

Modern TF/Keras 3 pipeline over the PlantVillage dataset (15 classes).

The steps below mirror `src/train.py`. For full training use the script:

```
python src/train.py --backbone custom --img-size 96 --epochs 20
```

In [ ]:
import os, json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), 'src')))
from preprocessing import DataLoader

In [ ]:
DATA = os.path.join('data', 'PlantVillage')
rows = []
for folder in sorted(os.listdir(DATA)):
    fp = os.path.join(DATA, folder)
    if not os.path.isdir(fp):
        continue
    for f in os.listdir(fp):
        if f.lower().endswith('.jpg'):
            rows.append({'path': os.path.join(fp, f), 'label': folder})
df = pd.DataFrame(rows)
print(f'Images: {len(df)}  Classes: {df.label.nunique()}')
df.label.value_counts()

In [ ]:
classes = sorted(df.label.unique())
label_map = {name: i for i, name in enumerate(classes)}
df['y'] = df.label.map(label_map)

rest, test = train_test_split(df, test_size=0.1, stratify=df['label'], random_state=42)
train, val = train_test_split(rest, test_size=0.1/0.9, stratify=rest['label'], random_state=42)
print(f'train {len(train)} | val {len(val)} | test {len(test)}')

In [ ]:
NUM_CLASSES = len(classes)
IMG = 96
BATCH = 64

train_ds = DataLoader(train.path.tolist(), train.y.tolist(), NUM_CLASSES, img_size=IMG, batch_size=BATCH, augment=True).get_dataset()
val_ds   = DataLoader(val.path.tolist(),   val.y.tolist(),   NUM_CLASSES, img_size=IMG, batch_size=BATCH).get_dataset()
test_ds  = DataLoader(test.path.tolist(),  test.y.tolist(),  NUM_CLASSES, img_size=IMG, batch_size=BATCH).get_dataset()

In [ ]:
from tensorflow.keras import layers, models

inp = layers.Input((IMG, IMG, 3))
x = layers.Conv2D(32, 3, padding='same')(inp); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
x = layers.Conv2D(64, 3, padding='same')(x);  x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.MaxPooling2D()(x)
x = layers.Conv2D(128, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.MaxPooling2D()(x)
x = layers.Conv2D(128, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.MaxPooling2D()(x)
x = layers.Conv2D(256, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = models.Model(inp, out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint('models/plant_disease_model.keras', monitor='val_accuracy', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
]

os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)
steps_train = int(np.ceil(len(train) / BATCH))
steps_val = int(np.ceil(len(val) / BATCH))

history = model.fit(train_ds, epochs=20, steps_per_epoch=steps_train, batch_size=BATCH,
                    validation_data=val_ds, validation_steps=steps_val, callbacks=callbacks, verbose=1)

best = tf.keras.models.load_model('models/plant_disease_model.keras')

In [ ]:
test_loss, test_acc = best.evaluate(test_ds, verbose=1)
print(f'Test accuracy: {test_acc:.4f} | loss: {test_loss:.4f}')

probs = best.predict(test_ds, verbose=1)
preds = np.argmax(probs, axis=1)
names = [classes[i] for i in range(len(classes))]
print(classification_report(test.y.tolist(), preds, target_names=names))

In [ ]:
from predict import PlantDiseasePredictor
p = PlantDiseasePredictor()

sample = test.sample(1, random_state=1).iloc[0]
result = p.predict(sample['path'])
print('True label    :', sample['label'])
print('Predicted     :', result['top_class'])
print('Confidence    :', f"{result['top_confidence']:.2%}")
print('Disease name  :', result['diagnosis']['common_name'])
print('Treatment     :', result['diagnosis']['treatment'])

## Summary

- **Pipeline**: tf.data -> augment -> model -> callbacks -> evaluation
- **Model**: Custom CNN (Conv + BatchNorm + MaxPool x4, GAP, Dense), 542k params
- **Backbone option**: `--backbone mobilenetv2` switches to transfer learning (needs GPU)
- **Accuracy target**: >= 97% test accuracy
- **Output**: top class + all-class probabilities + full diagnosis from `data/diagnosis.json`